# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [1]:
#Imports

import os
import json
import time
import gradio as gr

import whisper
from gtts import gTTS
import tempfile

from openai import OpenAI
import google.generativeai as genai

from dotenv import load_dotenv


In [2]:
#Load env variables

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")

if groq_api_key:
    print("Groq API Key loaded successfully")
else:
    print("Groq API Key is missing")

if google_api_key:
    print("Google API Key loaded successfully")
else:
    print("Google API Key is missing")

Groq API Key loaded successfully
Google API Key loaded successfully


In [3]:
#Connect to clients

groq_url = "https://api.groq.com/openai/v1"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

groq_client = OpenAI(
    api_key=groq_api_key,
    base_url=groq_url
)

gemini_client = OpenAI(
    api_key=google_api_key,
    base_url=gemini_url
)

print("Groq Client Connected")
print("Gemini Client Connected")

Groq Client Connected
Gemini Client Connected


In [5]:
#Define available models

MODELS = {

    # =====================================================
    # GROQ MODELS
    # =====================================================

    "Groq - Llama 70B": {
        "provider": "groq",
        "model": "llama-3.3-70b-versatile",
        "supports_tools": True
    },

    "Groq - Llama 8B": {
        "provider": "groq",
        "model": "llama-3.1-8b-instant",
        "supports_tools": True
    },

    # =====================================================
    # GEMINI MODELS
    # =====================================================

    "Gemini 2.5 Flash": {
        "provider": "gemini",
        "model": "gemini-2.5-flash",
        "supports_tools": False
    },

    "Gemini 2.5 Flash Lite": {
        "provider": "gemini",
        "model": "gemini-2.5-flash-lite",
        "supports_tools": False
    }
}

In [6]:
#System prompt

SYSTEM_PROMPT = """
You are an expert AI Technical Assistant.

You help users with:
- Programming
- AI/ML
- DSA
- Debugging
- System Design
- Web Development

Guidelines:
- Be accurate and concise.
- Explain concepts clearly.
- Use tools whenever required.
- If a tool can provide better information, prefer using the tool.
- Always behave like a senior software engineer mentor.

When tool information is available:
- incorporate it naturally
- explain results clearly
- avoid raw outputs
"""

In [7]:
#Tool Schema

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather information for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Name of the city"
                    }
                },
                "required": ["city"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "Get current local time of a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Name of the city"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

In [8]:
#Tool functions

from datetime import datetime

def get_weather(city):

    weather_database = {
        "kolkata": "32°C, Humid",
        "delhi": "38°C, Sunny",
        "mumbai": "30°C, Rainy",
        "london": "18°C, Cloudy",
        "new york": "20°C, Windy"
    }

    return weather_database.get(
        city.lower(),
        f"Weather data for {city} not found."
    )


def get_current_time(city):

    current_time = datetime.now().strftime("%H:%M:%S")

    return f"The current time in {city} is {current_time}"

In [9]:
#Tool Registry

available_tools = {
    "get_weather": get_weather,
    "get_current_time": get_current_time
}

In [10]:
#Main function

import json
import time

def generate_response(message, history, selected_model):

    print("\n" + "=" * 60)
    print("NEW REQUEST")
    print("=" * 60)

    model_info = MODELS[selected_model]

    provider = model_info["provider"]
    model_name = model_info["model"]
    supports_tools = model_info.get("supports_tools", False)

    print(f"\nSelected Model: {selected_model}")
    print(f"Provider: {provider}")
    print(f"Actual Model Name: {model_name}")
    print(f"Supports Tools: {supports_tools}")
    print(f"User Message: {message}")

    # =====================================================
    # MESSAGE HISTORY
    # =====================================================

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        }
    ]

    for user_msg, bot_msg in history:

        messages.append({
            "role": "user",
            "content": user_msg
        })

        messages.append({
            "role": "assistant",
            "content": bot_msg
        })

    messages.append({
        "role": "user",
        "content": message
    })

    # =====================================================
    # GROQ MODELS
    # =====================================================

    if provider == "groq":

        print("\nUsing Groq Model")

        try:

            # =============================================
            # TOOL CALLING ENABLED
            # =============================================

            if supports_tools:

                print("\nSending request with tool support...")

                first_response = groq_client.chat.completions.create(
                    model=model_name,
                    messages=messages,
                    tools=TOOLS,
                    tool_choice="auto"
                )

                response_message = first_response.choices[0].message

                tool_calls = response_message.tool_calls

                # =========================================
                # TOOL CALL DETECTED
                # =========================================

                if tool_calls:

                    print("\nTOOL CALL DETECTED")

                    messages.append(response_message)

                    for tool_call in tool_calls:

                        function_name = tool_call.function.name

                        print(f"\nTool Name: {function_name}")

                        function_to_call = available_tools[
                            function_name
                        ]

                        function_args = json.loads(
                            tool_call.function.arguments
                        )

                        print(
                            f"Tool Arguments: {function_args}"
                        )

                        function_response = function_to_call(
                            **function_args
                        )

                        print(
                            f"Tool Response: {function_response}"
                        )

                        messages.append({
                            "role": "tool",
                            "tool_call_id": tool_call.id,
                            "name": function_name,
                            "content": function_response
                        })

                    print(
                        "\nSending tool result back to model..."
                    )

                    second_response = groq_client.chat.completions.create(
                        model=model_name,
                        messages=messages,
                        stream=True
                    )

                    partial_response = ""

                    print("\nStreaming Final Response...\n")

                    for chunk in second_response:

                        content = (
                            chunk
                            .choices[0]
                            .delta
                            .content
                        )

                        if content:

                            partial_response += content

                            yield partial_response

                    print("\nResponse Completed")

                # =========================================
                # NORMAL RESPONSE (NO TOOL CALL)
                # =========================================

                else:

                    print("\nNo tool call needed")

                    stream = groq_client.chat.completions.create(
                        model=model_name,
                        messages=messages,
                        stream=True
                    )

                    partial_response = ""

                    print("\nStreaming Normal Response...\n")

                    for chunk in stream:

                        content = (
                            chunk
                            .choices[0]
                            .delta
                            .content
                        )

                        if content:

                            partial_response += content

                            yield partial_response

                    print("\nResponse Completed")

            # =============================================
            # NO TOOL SUPPORT
            # =============================================

            else:

                print("\nThis Groq model does not support tools")

                stream = groq_client.chat.completions.create(
                    model=model_name,
                    messages=messages,
                    stream=True
                )

                partial_response = ""

                print("\nStreaming Response...\n")

                for chunk in stream:

                    content = (
                        chunk
                        .choices[0]
                        .delta
                        .content
                    )

                    if content:

                        partial_response += content

                        yield partial_response

                print("\nResponse Completed")

        except Exception as e:

            print("\nERROR:")
            print(str(e))

            yield f"Error: {str(e)}"

    # =====================================================
    # GEMINI MODELS
    # =====================================================

    elif provider == "gemini":

        print("\nUsing Gemini Model")
        print(f"Gemini Model Name: {model_name}")
        print(f"User Query: {message}")

        try:

            model = genai.GenerativeModel(model_name)

            prompt = (
                SYSTEM_PROMPT
                + "\n\nUser: "
                + message
            )

            print("\nSending request to Gemini...")

            # =============================================
            # REAL GEMINI STREAMING
            # =============================================

            response = model.generate_content(
                prompt,
                stream=True
            )

            partial_response = ""

            print("\nStreaming Gemini Response...\n")

            for chunk in response:

                try:

                    content = chunk.text

                    if content:

                        partial_response += content

                        yield partial_response

                except:
                    pass

            print("\nResponse Completed")

        except Exception as e:

            print("\nERROR:")
            print(str(e))

            yield f"Error: {str(e)}"

In [ ]:
#Gradio UI

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        """
        # 🤖 AI Technical Assistant

        ### Features
        - Multi-Model Support
        - Groq + Gemini + OpenRouter
        - Real Tool Calling
        - Streaming Responses
        - System Prompting

        ### Example Questions
        - What is the weather in Kolkata?
        - Tell me the current time in Delhi
        - Explain transformers in AI
        - Debug my Python code
        """
    )

    model_dropdown = gr.Dropdown(
        choices=list(MODELS.keys()),
        value="Groq - Llama 70B",
        label="Choose Model"
    )

    chatbot = gr.ChatInterface(
        fn=generate_response,
        additional_inputs=[model_dropdown]
    )

demo.launch(share=True)

c:\Users\KIIT\projects\llm_engineering\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7867
* Running on public URL: https://66ab7438c11d3edaeb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



NEW REQUEST

Selected Model: Groq - Llama 70B
Provider: groq
Actual Model Name: llama-3.3-70b-versatile
Supports Tools: True

User Message: What is the weather of Kolkata?

Sending request with tools enabled...

TOOL CALL DETECTED

Tool Name: get_weather
Tool Arguments: {'city': 'Kolkata'}
Tool Response: 32°C, Humid

Sending tool result back to model...

Response Completed

NEW REQUEST

Selected Model: Gemini 2.5 Flash
Provider: gemini
Actual Model Name: gemini-2.5-flash
Supports Tools: False

User Message: Explain transformers to a layman

Using Gemini Model

Streaming Gemini Response...


Response Completed


In [11]:
import os

os.environ["PATH"] += os.pathsep + r"C:\ffmpeg\ffmpeg-8.1.1-essentials_build\bin"

print("FFmpeg PATH Added")

FFmpeg PATH Added


In [12]:
#Load whisper model

import whisper

whisper_model = whisper.load_model("base")

print("Whisper Loaded")

Whisper Loaded


In [13]:
#Speech-to-text function

def transcribe_audio(audio_path):

    print("\nTranscribing Audio...", flush=True)

    result = whisper_model.transcribe(audio_path)

    text = result["text"]

    print(f"Transcribed Text: {text}", flush=True)

    return text

In [14]:
#Text-to-speech function

def text_to_speech(text):

    print("\nGenerating Voice Response...", flush=True)

    tts = gTTS(
        text=text,
        lang="en"
    )

    temp_audio = tempfile.NamedTemporaryFile(
        delete=False,
        suffix=".mp3"
    )

    tts.save(temp_audio.name)

    print("Voice Response Ready", flush=True)

    return temp_audio.name

In [15]:
#Wrapper functions

import re

# =====================================================
# CLEAN TEXT FOR SPEECH
# =====================================================

def clean_text_for_tts(text):

    # Remove code blocks
    text = re.sub(
        r'```.*?```',
        '',
        text,
        flags=re.DOTALL
    )

    # Remove markdown symbols
    text = re.sub(
        r'[#*_`>-]',
        '',
        text
    )

    # Remove URLs
    text = re.sub(
        r'http\\S+',
        '',
        text
    )

    # Remove multiple spaces/newlines
    text = re.sub(
        r'\\s+',
        ' ',
        text
    )

    return text.strip()


# =====================================================
# SPEECH → TEXT RESPONSE
# =====================================================

def speech_to_text_chat(audio, history, selected_model):

    print("\nSpeech-to-Text Mode", flush=True)

    # ==========================================
    # AUDIO → TEXT
    # ==========================================

    user_text = transcribe_audio(audio)

    final_response = ""

    # ==========================================
    # GENERATE AI RESPONSE
    # ==========================================

    for chunk in generate_response(
        user_text,
        history,
        selected_model
    ):

        final_response = chunk

    return (
        user_text,
        final_response
    )


# =====================================================
# TEXT → VOICE RESPONSE
# =====================================================

def text_to_voice_chat(message, history, selected_model):

    print("\nText-to-Voice Mode", flush=True)

    final_response = ""

    # ==========================================
    # GENERATE AI RESPONSE
    # ==========================================

    for chunk in generate_response(
        message,
        history,
        selected_model
    ):

        final_response = chunk

    # ==========================================
    # CLEAN RESPONSE FOR SPEECH
    # ==========================================

    clean_response = clean_text_for_tts(
        final_response
    )

    print(
        f"\nCleaned Response For TTS:\n{clean_response}",
        flush=True
    )

    # ==========================================
    # TEXT → AUDIO
    # ==========================================

    audio_response = text_to_speech(
        clean_response
    )

    return audio_response

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        """
        # 🤖 AI Technical Assistant

        ## Features
        - Groq + Gemini Models
        - Tool Calling
        - Streaming
        - Speech-to-Text
        - Text-to-Speech
        - Multi-Model Switching
        """
    )

    # =====================================================
    # MODEL SELECTOR
    # =====================================================

    model_dropdown = gr.Dropdown(
        choices=list(MODELS.keys()),
        value="Groq - Llama 70B",
        label="Choose Model"
    )

    # =====================================================
    # TEXT CHAT TAB
    # =====================================================

    with gr.Tab("💬 Text Chat"):

        chatbot = gr.Chatbot(
            height=500
        )

        msg = gr.Textbox(
            label="Enter Message"
        )

        clear = gr.Button("Clear Chat")

        def chat_response(message, chat_history, selected_model):

            bot_message = ""

            for chunk in generate_response(
                message,
                chat_history,
                selected_model
            ):

                bot_message = chunk

            chat_history.append(
                (message, bot_message)
            )

            return "", chat_history

        msg.submit(
            fn=chat_response,
            inputs=[
                msg,
                chatbot,
                model_dropdown
            ],
            outputs=[
                msg,
                chatbot
            ]
        )

        clear.click(
            lambda: None,
            None,
            chatbot,
            queue=False
        )

    # =====================================================
    # SPEECH → TEXT TAB
    # =====================================================

    with gr.Tab("🎤 Speech → Text Response"):

        audio_input = gr.Audio(
            sources=["microphone"],
            type="filepath",
            streaming=False,
            label="🎤 Speak Your Question"
        )

        speech_text = gr.Textbox(
            label="Transcribed Question"
        )

        speech_response = gr.Textbox(
            label="AI Text Response"
        )

        speech_btn = gr.Button(
            "Generate Text Response"
        )

        speech_btn.click(
            fn=speech_to_text_chat,
            inputs=[
                audio_input,
                chatbot,
                model_dropdown
            ],
            outputs=[
                speech_text,
                speech_response
            ]
        )

    # =====================================================
    # TEXT → VOICE TAB
    # =====================================================

    with gr.Tab("🔊 Text → Voice Response"):

        text_input = gr.Textbox(
            label="Enter Your Question"
        )

        audio_output = gr.Audio(
            label="AI Voice Response"
        )

        voice_btn = gr.Button(
            "Generate Voice Response"
        )

        voice_btn.click(
            fn=text_to_voice_chat,
            inputs=[
                text_input,
                chatbot,
                model_dropdown
            ],
            outputs=audio_output
        )

demo.launch(
    share=True,
    debug=True,
    inline=False,
    inbrowser=True
)

C:\Users\KIIT\AppData\Local\Temp\ipykernel_21948\1248429681.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://11b9119017eeeb00b3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)

Speech-to-Text Mode

Transcribing Audio...


c:\Users\KIIT\projects\llm_engineering\.venv\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcribed Text:  What is AI?

NEW REQUEST

Selected Model: Groq - Llama 70B
Provider: groq
Actual Model Name: llama-3.3-70b-versatile
Supports Tools: True
User Message:  What is AI?

Using Groq Model

Sending request with tool support...

No tool call needed

Streaming Normal Response...


Response Completed

Text-to-Voice Mode

NEW REQUEST

Selected Model: Groq - Llama 8B
Provider: groq
Actual Model Name: llama-3.1-8b-instant
Supports Tools: True
User Message: What is transformers?

Using Groq Model

Sending request with tool support...

ERROR:
Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=brave_search>Transformers are a type of neural network architecture in deep learning that is particularly well-suited for natural language processing (NLP) tasks. They were first introduced in 2017 by Vaswani et al.